In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import os
import numpy as np
import json
from collections import Counter
from numpy import linalg as la
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import OPTICS
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score
import importlib
import model as model_module
importlib.reload(model_module)
LogKG = model_module.LogKG

# OS_preprocessed dataset config
OS_PREPROCESSED_DIR = os.path.join("..", "data", "OS_preprocessed")
OS_CASE_DIR = os.path.join(OS_PREPROCESSED_DIR, "cases")
OS_CONFIG_MAJOR_PATH = os.path.join(OS_PREPROCESSED_DIR, "config_major.json")
OS_CONFIG_MINOR_PATH = os.path.join(OS_PREPROCESSED_DIR, "config_minor.json")
OS_CONFIG_PAIR_PATH = os.path.join(OS_PREPROCESSED_DIR, "config_pair.json")

from sklearn.feature_extraction.text import HashingVectorizer




In [2]:
# Core experiment config
EMBEDDING_SIZE = 16
template_embedding = None

# Label granularity for OS_preprocessed training:
# - 'major': major_problem_type (recommended first stage)
# - 'minor': minor_problem_type (hard, long-tail)
# - 'pair' : major||minor
LABEL_LEVEL = "major"

# Template embedding mode:
# - 'random'   : random vectors (current stable baseline)
# - 'hash_text': hash EventTemplate text to dense vectors
TEMPLATE_EMBEDDING_MODE = "random"

# Eval mode:
# - 'supervised': train RF with true labels directly (recommended)
# - 'logkg'     : original LogKG pseudo-label pipeline (cluster -> RF)
EVAL_MODE = "supervised"

# Rare class handling for StratifiedKFold
MIN_SAMPLES_PER_CLASS = 2
RARE_CLASS_STRATEGY = "drop"



In [3]:
# ===============================
# ???? case ??????ground truth label?
# ===============================
def get_case_truth_label(case_name_list, input_config):

    # ???????????? case ???
    truth_label_list = np.zeros((len(case_name_list)), dtype=int)

    # ?? case_name -> ?? ??????? list.index ? O(n^2)
    case_index_map = {name: idx for idx, name in enumerate(case_name_list)}

    # ???? ????? -> ????? ???
    label_config = {}

    # ?? input_config ????????
    for index, fault_class in enumerate(input_config.keys()):
        label_config[fault_class] = index

        # ??????? case ??????
        for case_name in input_config[fault_class]:
            if case_name not in case_index_map:
                raise ValueError(f"case in config but missing csv file: {case_name}")
            truth_label_list[case_index_map[case_name]] = index

    return truth_label_list



In [4]:
# ===============================
# 聚类中一个簇允许的最少样本数量
# ===============================
# 如果一个簇里的样本数 < min_samples
# OPTICS 可能会把这些点当作噪声点（label = -1）
min_samples = 3


# ===============================
# 构建聚类模型（Cluster Model）
# ===============================
def build_cluster_model():
    return OPTICS(
        min_samples=min_samples,   # 一个簇至少包含的样本数
        metric="cosine",           # 使用余弦距离（cosine distance）计算样本相似度
        xi=0.05,                   # OPTICS 中用于检测簇边界的参数，越小簇越细
        algorithm="brute"          # 使用暴力搜索计算距离（适合小规模数据）
    )


# ===============================
# IDF 阈值（过滤低信息模板）
# ===============================
# 在 LogKG 中：
# IDF = log10( case总数 / 包含该模板的case数 )

# 如果 IDF < IDF_threshold
# 则认为该模板信息量太低（例如常见日志）
# 会被过滤掉（权重设为0）

IDF_threshold = 0.4


# ===============================
# 分类器（Classifier）
# ===============================
# LogKG 在聚类后会生成 pseudo-label
# 然后用一个监督分类器进行训练

clf = RandomForestClassifier


# 分类器参数

clf_kwargs = {
    "random_state": 42,
    "n_estimators": 600,
    "class_weight": "balanced_subsample",
    "n_jobs": 1,
}




In [5]:
# =========================
# 1) 计算 EDM: Euclidean Distance Matrix（欧氏距离矩阵）
# 输入:
#   X: shape=(n_samples, n_features)
# 输出:
#   D: shape=(n_samples, n_samples)
#       D[i, j] 表示样本 i 和样本 j 的“欧氏距离”
# 注意:
#   函数名叫 compute_squared_EDM_method，但这里并没有平方（不是 squared distance）
#   la.norm 默认是 L2 范数 => sqrt(sum((xi-xj)^2))
# =========================
def compute_squared_EDM_method(X):
    n, m = X.shape
    D = np.zeros([n, n])

    # 双重循环，计算上三角（i < j），再对称赋值到下三角
    for i in range(n):
        for j in range(i + 1, n):
            # 计算两点欧氏距离
            D[i, j] = la.norm(X[i, :] - X[j, :])
            D[j, i] = D[i, j]
    return D


# =========================
# 2) 找“类中心点”的索引（类似 medoid）
# 思路:
#   - 先算簇内所有点两两距离矩阵 D
#   - 对每个点，把它到其它点的距离求和（越小越“居中”）
#   - 返回距离和最小的那个点的下标
# 输入:
#   cluster_embedding: shape=(k, embedding_dim)，某个簇里的 k 个点
# 输出:
#   返回簇内的“中心点”在 cluster_embedding 里的行号（0~k-1）
# =========================
def get_centroid_index(cluster_embedding):
    # 对每一行求和: distance_array[i] = sum_j D[i, j]
    distance_array = np.sum(compute_squared_EDM_method(cluster_embedding), axis=1)
    # 返回距离和最小的点（簇内最代表性的样本）
    return np.argmin(distance_array)


# =========================
# 3) 训练集聚类 + 选每个簇的代表样本（centroid/medoid）
# 输入:
#   train_set: 当前 fold 的训练 embedding（shape=(n_train, emb_dim)）
#   train_index: 这些训练样本在“全数据”里的原始索引列表
# 输出:
#   centroids: 每个簇选出来的代表样本的“原始索引”（对齐全数据）
#   cluster_: 聚类结果，长度 n_train，每个元素是簇 id（0..class_num-1），可能会有 -1 噪声
# 注意:
#   - 代码里把 -1 当噪声，但 class_num = max(cluster_)+1 会忽略 -1（不计为类别）
#   - 若存在 -1，后续 list comprehension 只遍历 0..class_num-1，不会处理噪声点
# =========================
def get_logkg_result(train_set, train_index, verbose=False):
    cluster_model = build_cluster_model()
    # 聚类：返回每个样本所属簇编号
    cluster_ = cluster_model.fit_predict(train_set)

    # 类别数量（不包含 -1 噪声）
    class_num = np.max(cluster_) + 1

    if verbose:
        print(cluster_)
        print(Counter(cluster_))
        print("class_num: {}".format(class_num))

    # 对每个簇 i：
    #   1) 找出 train_set 中属于该簇的样本下标集合 idx_in_train
    #   2) 在该簇内部找“中心点”（簇内距离和最小）
    #   3) 把它映射回全数据索引 train_index[...]
    centroids = [
        train_index[
            np.where(cluster_ == i)[0][
                get_centroid_index(train_set[np.where(cluster_ == i)[0]])
            ]
        ]
        for i in range(class_num)
    ]
    return centroids, cluster_


# =========================
# 4) 单个 fold 的训练/测试流程
# 流程:
#   1) 用 LogKG 计算 train/test 的 case embedding（每个 case 一个向量）
#   2) 在 train embedding 上聚类
#   3) 每个簇选一个代表样本，用其真实标签当作该簇的标签（pseudo-label）
#   4) 用这些 pseudo-label 样本训练一个分类器（clf）
#   5) 用分类器预测 test，返回 acc / macro_f1
# =========================
def LogKG_exp_run(case_name_list, case_truth_label,
                  train_index, test_index,
                  logkg_config, verbose=False):
    logkg = logkg_config

    # split 不能为空
    if len(train_index) == 0 or len(test_index) == 0:
        raise ValueError("train_index/test_index is empty. Please set a non-empty split before running.")

    # 生成训练/测试的 embedding 字典
    # train_embedding_dict: {case_name: embedding_vector}
    # test_embedding_dict: {case_name: embedding_vector}
    logkg.get_train_embedding()
    logkg.get_test_embedding()
    train_embedding = logkg.train_embedding_dict
    test_embedding = logkg.test_embedding_dict

    # 组装当前 fold 的 train/test 矩阵
    train_set = np.array([train_embedding[case_name_list[index]] for index in train_index])
    test_set = np.array([test_embedding[case_name_list[index]] for index in test_index])

    # 训练集聚类 + 每个簇选代表样本（返回的是“全数据索引”）
    cluster_centroids, cluster_result = get_logkg_result(
        train_set=train_set,
        train_index=train_index,
        verbose=verbose
    )

    # classify_index：给训练集每个样本分配一个“簇标签对应的真实类别”
    # 初始化为 -1 表示未分配/噪声
    classify_index = np.zeros(len(cluster_result), dtype=int) - 1

    # 遍历每个簇 i，把簇里所有样本的 pseudo-label 设置为该簇代表样本的真实标签
    for i in range(np.max(cluster_result) + 1):
        class_label = case_truth_label[cluster_centroids[i]]
        classify_index[np.where(cluster_result == i)[0]] = class_label

    # 只保留被赋值成功的样本（!= -1），构造监督训练数据
    classify_train_mask = classify_index != -1
    if not np.any(classify_train_mask):
        raise ValueError("No labeled samples after clustering.")

    classify_train_label = classify_index[classify_train_mask]
    classify_train_set = train_set[classify_train_mask]

    # 训练分类器（clf 和 clf_kwargs 是外部定义的）
    classifier = clf(**clf_kwargs)
    classifier.fit(classify_train_set, classify_train_label)

    # 测试集评估
    y_true = case_truth_label[test_index].astype(int)
    y_pred = classifier.predict(test_set).astype(int)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return y_true, y_pred, acc, macro_f1


# =========================
# 5) StratifiedKFold（分层K折）评估
# 关键点:
#   - 如果某个类别样本很少，n_splits 不能超过最小类别数
#   - 每折里重新构造 train_df/test_df（按 case_name -> log_df）
#   - 每折训练 LogKG 并评估，最后汇总均值/方差与混淆矩阵 confusion_matrix
# =========================
def run_stratified_kfold_eval(case_name_list, case_truth_label,
                              case_log_df, template_embedding,
                              n_splits=5, random_state=42,
                              verbose_fold=False):
    # 统计每类数量，避免分层K折时某类样本数 < n_splits
    class_counts = Counter(case_truth_label.tolist())
    min_class_count = min(class_counts.values())
    effective_splits = min(n_splits, min_class_count)
    if effective_splits < 2:
        raise ValueError(f"Not enough samples per class for StratifiedKFold: min_class_count={min_class_count}")

    skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=random_state)

    acc_list = []
    f1_list = []
    y_true_all = []
    y_pred_all = []

    # 逐 fold 训练与测试
    for fold_idx, (train_index, test_index) in enumerate(
        skf.split(np.arange(len(case_name_list)), case_truth_label),
        start=1
    ):
        train_index = train_index.tolist()
        test_index = test_index.tolist()

        # 构造当前 fold 的训练/测试日志字典
        # train_df/test_df: {case_name: 对应的日志DataFrame}
        train_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in train_index}
        test_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in test_index}

        # 初始化 LogKG（兼容不同构造函数签名）
        try:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding, embedding_size=EMBEDDING_SIZE)
        except TypeError:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding)

        # 跑一折实验
        y_true, y_pred, acc, macro_f1 = LogKG_exp_run(
            case_name_list=case_name_list,
            case_truth_label=case_truth_label,
            train_index=train_index,
            test_index=test_index,
            logkg_config=model,
            verbose=verbose_fold,
        )

        acc_list.append(acc)
        f1_list.append(macro_f1)
        y_true_all.append(y_true)
        y_pred_all.append(y_pred)

        print(f"fold {fold_idx}/{effective_splits}: acc={acc:.4f}, macro_f1={macro_f1:.4f}, test_size={len(test_index)}")

    # 汇总所有 fold 的预测结果
    y_true_all = np.concatenate(y_true_all).astype(int)
    y_pred_all = np.concatenate(y_pred_all).astype(int)

    acc_mean = float(np.mean(acc_list))
    acc_std = float(np.std(acc_list))
    f1_mean = float(np.mean(f1_list))
    f1_std = float(np.std(f1_list))

    # 混淆矩阵：labels 用真实标签集合（保持顺序一致）
    labels = np.unique(case_truth_label)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

    print("=" * 30, "StratifiedKFold Summary", "=" * 30)
    print(f"accuracy: mean={acc_mean:.4f}, std={acc_std:.4f}")
    print(f"macro_f1: mean={f1_mean:.4f}, std={f1_std:.4f}")
    print("confusion_matrix labels:", labels.tolist())
    print(cm)

    # 返回结构化结果，便于外部保存/画图
    return {
        "acc_list": acc_list,
        "f1_list": f1_list,
        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "f1_mean": f1_mean,
        "f1_std": f1_std,
        "labels": labels,
        "confusion_matrix": cm,
    }


# =========================
# 6) StratifiedKFold???K?????OS????
# ????:
#   - ?????? < min_samples_per_class ???????????????
#   - ?????????????
# =========================
def run_stratified_kfold_eval(case_name_list, case_truth_label,
                              case_log_df, template_embedding,
                              n_splits=5, random_state=42,
                              verbose_fold=False,
                              min_samples_per_class=2,
                              rare_class_strategy="drop"):
    if len(case_name_list) != len(case_truth_label):
        raise ValueError("case_name_list and case_truth_label length mismatch")

    class_counts = Counter(case_truth_label.tolist())
    rare_labels = [label for label, count in class_counts.items() if count < min_samples_per_class]

    if rare_labels:
        if rare_class_strategy == "drop":
            keep_indices = [idx for idx, y in enumerate(case_truth_label.tolist()) if y not in rare_labels]
            dropped_count = len(case_name_list) - len(keep_indices)

            if len(keep_indices) == 0:
                raise ValueError("All samples are dropped by rare class filtering.")

            case_name_list = [case_name_list[idx] for idx in keep_indices]
            case_truth_label = case_truth_label[keep_indices]
            case_log_df = {name: case_log_df[name] for name in case_name_list}

            class_counts = Counter(case_truth_label.tolist())
            print(
                f"[Warn] Dropped {dropped_count} samples from rare classes "
                f"(labels={rare_labels}, min_samples_per_class={min_samples_per_class})."
            )
            print(f"[Info] Remaining class distribution: {class_counts}")
        else:
            raise ValueError(
                f"Not enough samples per class for StratifiedKFold: "
                f"rare_labels={rare_labels}, min_samples_per_class={min_samples_per_class}"
            )

    min_class_count = min(class_counts.values())
    effective_splits = min(n_splits, min_class_count)
    if effective_splits < 2:
        raise ValueError(
            f"Not enough samples per class for StratifiedKFold after filtering: "
            f"min_class_count={min_class_count}."
        )

    skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=random_state)

    acc_list = []
    f1_list = []
    y_true_all = []
    y_pred_all = []

    for fold_idx, (train_index, test_index) in enumerate(
        skf.split(np.arange(len(case_name_list)), case_truth_label),
        start=1
    ):
        train_index = train_index.tolist()
        test_index = test_index.tolist()

        train_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in train_index}
        test_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in test_index}

        try:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding, embedding_size=EMBEDDING_SIZE)
        except TypeError:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding)

        y_true, y_pred, acc, macro_f1 = LogKG_exp_run(
            case_name_list=case_name_list,
            case_truth_label=case_truth_label,
            train_index=train_index,
            test_index=test_index,
            logkg_config=model,
            verbose=verbose_fold,
        )

        acc_list.append(acc)
        f1_list.append(macro_f1)
        y_true_all.append(y_true)
        y_pred_all.append(y_pred)

        print(f"fold {fold_idx}/{effective_splits}: acc={acc:.4f}, macro_f1={macro_f1:.4f}, test_size={len(test_index)}")

    y_true_all = np.concatenate(y_true_all).astype(int)
    y_pred_all = np.concatenate(y_pred_all).astype(int)

    acc_mean = float(np.mean(acc_list))
    acc_std = float(np.std(acc_list))
    f1_mean = float(np.mean(f1_list))
    f1_std = float(np.std(f1_list))

    labels = np.unique(case_truth_label)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

    print("=" * 30, "StratifiedKFold Summary", "=" * 30)
    print(f"accuracy: mean={acc_mean:.4f}, std={acc_std:.4f}")
    print(f"macro_f1: mean={f1_mean:.4f}, std={f1_std:.4f}")
    print("confusion_matrix labels:", labels.tolist())
    print(cm)

    return {
        "acc_list": acc_list,
        "f1_list": f1_list,
        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "f1_mean": f1_mean,
        "f1_std": f1_std,
        "labels": labels,
        "confusion_matrix": cm,
        "effective_splits": effective_splits,
        "final_class_dist": dict(class_counts),
    }




def run_supervised_kfold_eval(case_name_list, case_truth_label,
                              case_log_df, template_embedding,
                              n_splits=5, random_state=42,
                              min_samples_per_class=2,
                              rare_class_strategy="drop"):
    if len(case_name_list) != len(case_truth_label):
        raise ValueError("case_name_list and case_truth_label length mismatch")

    class_counts = Counter(case_truth_label.tolist())
    rare_labels = [label for label, count in class_counts.items() if count < min_samples_per_class]

    if rare_labels:
        if rare_class_strategy == "drop":
            keep_indices = [idx for idx, y in enumerate(case_truth_label.tolist()) if y not in rare_labels]
            dropped_count = len(case_name_list) - len(keep_indices)
            case_name_list = [case_name_list[idx] for idx in keep_indices]
            case_truth_label = case_truth_label[keep_indices]
            case_log_df = {name: case_log_df[name] for name in case_name_list}
            class_counts = Counter(case_truth_label.tolist())
            print(
                f"[Warn] Dropped {dropped_count} samples from rare classes "
                f"(labels={rare_labels}, min_samples_per_class={min_samples_per_class})."
            )
            print(f"[Info] Remaining class distribution: {class_counts}")
        else:
            raise ValueError(
                f"Not enough samples per class for StratifiedKFold: "
                f"rare_labels={rare_labels}, min_samples_per_class={min_samples_per_class}"
            )

    min_class_count = min(class_counts.values())
    effective_splits = min(n_splits, min_class_count)
    if effective_splits < 2:
        raise ValueError(
            f"Not enough samples per class for StratifiedKFold after filtering: "
            f"min_class_count={min_class_count}."
        )

    skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=random_state)

    acc_list, f1_list = [], []
    y_true_all, y_pred_all = [], []

    for fold_idx, (train_index, test_index) in enumerate(
        skf.split(np.arange(len(case_name_list)), case_truth_label), start=1
    ):
        train_index = train_index.tolist()
        test_index = test_index.tolist()

        train_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in train_index}
        test_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in test_index}

        try:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding, embedding_size=EMBEDDING_SIZE)
        except TypeError:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding)

        model.get_train_embedding()
        model.get_test_embedding()

        train_set = np.array([model.train_embedding_dict[case_name_list[idx]] for idx in train_index])
        test_set = np.array([model.test_embedding_dict[case_name_list[idx]] for idx in test_index])

        y_train = case_truth_label[train_index].astype(int)
        y_true = case_truth_label[test_index].astype(int)

        classifier = clf(**clf_kwargs)
        classifier.fit(train_set, y_train)
        y_pred = classifier.predict(test_set).astype(int)

        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro")

        acc_list.append(acc)
        f1_list.append(macro_f1)
        y_true_all.append(y_true)
        y_pred_all.append(y_pred)

        print(f"fold {fold_idx}/{effective_splits}: acc={acc:.4f}, macro_f1={macro_f1:.4f}, test_size={len(test_index)}")

    y_true_all = np.concatenate(y_true_all).astype(int)
    y_pred_all = np.concatenate(y_pred_all).astype(int)

    acc_mean = float(np.mean(acc_list))
    acc_std = float(np.std(acc_list))
    f1_mean = float(np.mean(f1_list))
    f1_std = float(np.std(f1_list))

    labels = np.unique(case_truth_label)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

    print("=" * 30, "Supervised StratifiedKFold Summary", "=" * 30)
    print(f"accuracy: mean={acc_mean:.4f}, std={acc_std:.4f}")
    print(f"macro_f1: mean={f1_mean:.4f}, std={f1_std:.4f}")
    print("confusion_matrix labels:", labels.tolist())
    print(cm)

    return {
        "acc_list": acc_list,
        "f1_list": f1_list,
        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "f1_mean": f1_mean,
        "f1_std": f1_std,
        "labels": labels,
        "confusion_matrix": cm,
        "effective_splits": effective_splits,
        "final_class_dist": dict(class_counts),
    }

# ===== Metrics Extension Override =====
def _calc_eval_metrics(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
    }


def run_stratified_kfold_eval(case_name_list, case_truth_label,
                              case_log_df, template_embedding,
                              n_splits=5, random_state=42,
                              verbose_fold=False,
                              min_samples_per_class=2,
                              rare_class_strategy="drop"):
    if len(case_name_list) != len(case_truth_label):
        raise ValueError("case_name_list and case_truth_label length mismatch")

    class_counts = Counter(case_truth_label.tolist())
    rare_labels = [label for label, count in class_counts.items() if count < min_samples_per_class]

    if rare_labels:
        if rare_class_strategy == "drop":
            keep_indices = [idx for idx, y in enumerate(case_truth_label.tolist()) if y not in rare_labels]
            dropped_count = len(case_name_list) - len(keep_indices)
            if len(keep_indices) == 0:
                raise ValueError("All samples are dropped by rare class filtering.")
            case_name_list = [case_name_list[idx] for idx in keep_indices]
            case_truth_label = case_truth_label[keep_indices]
            case_log_df = {name: case_log_df[name] for name in case_name_list}
            class_counts = Counter(case_truth_label.tolist())
            print(
                f"[Warn] Dropped {dropped_count} samples from rare classes "
                f"(labels={rare_labels}, min_samples_per_class={min_samples_per_class})."
            )
            print(f"[Info] Remaining class distribution: {class_counts}")
        else:
            raise ValueError(
                f"Not enough samples per class for StratifiedKFold: "
                f"rare_labels={rare_labels}, min_samples_per_class={min_samples_per_class}"
            )

    min_class_count = min(class_counts.values())
    effective_splits = min(n_splits, min_class_count)
    if effective_splits < 2:
        raise ValueError(
            f"Not enough samples per class for StratifiedKFold after filtering: "
            f"min_class_count={min_class_count}."
        )

    skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=random_state)

    acc_list, macro_f1_list = [], []
    weighted_f1_list, macro_recall_list, weighted_recall_list = [], [], []
    y_true_all, y_pred_all = [], []

    for fold_idx, (train_index, test_index) in enumerate(
        skf.split(np.arange(len(case_name_list)), case_truth_label),
        start=1,
    ):
        train_index = train_index.tolist()
        test_index = test_index.tolist()

        train_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in train_index}
        test_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in test_index}

        try:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding, embedding_size=EMBEDDING_SIZE)
        except TypeError:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding)

        y_true, y_pred, _, _ = LogKG_exp_run(
            case_name_list=case_name_list,
            case_truth_label=case_truth_label,
            train_index=train_index,
            test_index=test_index,
            logkg_config=model,
            verbose=verbose_fold,
        )

        m = _calc_eval_metrics(y_true, y_pred)
        acc_list.append(m["acc"])
        macro_f1_list.append(m["macro_f1"])
        weighted_f1_list.append(m["weighted_f1"])
        macro_recall_list.append(m["macro_recall"])
        weighted_recall_list.append(m["weighted_recall"])
        y_true_all.append(y_true)
        y_pred_all.append(y_pred)

        print(
            f"fold {fold_idx}/{effective_splits}: "
            f"acc={m['acc']:.4f}, macro_f1={m['macro_f1']:.4f}, weighted_f1={m['weighted_f1']:.4f}, "
            f"macro_recall={m['macro_recall']:.4f}, weighted_recall={m['weighted_recall']:.4f}, test_size={len(test_index)}"
        )

    y_true_all = np.concatenate(y_true_all).astype(int)
    y_pred_all = np.concatenate(y_pred_all).astype(int)

    acc_mean, acc_std = float(np.mean(acc_list)), float(np.std(acc_list))
    macro_f1_mean, macro_f1_std = float(np.mean(macro_f1_list)), float(np.std(macro_f1_list))
    weighted_f1_mean, weighted_f1_std = float(np.mean(weighted_f1_list)), float(np.std(weighted_f1_list))
    macro_recall_mean, macro_recall_std = float(np.mean(macro_recall_list)), float(np.std(macro_recall_list))
    weighted_recall_mean, weighted_recall_std = float(np.mean(weighted_recall_list)), float(np.std(weighted_recall_list))

    labels = np.unique(case_truth_label)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

    print("=" * 30, "StratifiedKFold Summary", "=" * 30)
    print(f"accuracy: mean={acc_mean:.4f}, std={acc_std:.4f}")
    print(f"macro_f1: mean={macro_f1_mean:.4f}, std={macro_f1_std:.4f}")
    print(f"weighted_f1: mean={weighted_f1_mean:.4f}, std={weighted_f1_std:.4f}")
    print(f"macro_recall: mean={macro_recall_mean:.4f}, std={macro_recall_std:.4f}")
    print(f"weighted_recall: mean={weighted_recall_mean:.4f}, std={weighted_recall_std:.4f}")
    print("confusion_matrix labels:", labels.tolist())
    print(cm)

    return {
        "acc_list": acc_list,
        "macro_f1_list": macro_f1_list,
        "f1_list": macro_f1_list,
        "weighted_f1_list": weighted_f1_list,
        "macro_recall_list": macro_recall_list,
        "weighted_recall_list": weighted_recall_list,
        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "macro_f1_mean": macro_f1_mean,
        "f1_mean": macro_f1_mean,
        "macro_f1_std": macro_f1_std,
        "f1_std": macro_f1_std,
        "weighted_f1_mean": weighted_f1_mean,
        "weighted_f1_std": weighted_f1_std,
        "macro_recall_mean": macro_recall_mean,
        "macro_recall_std": macro_recall_std,
        "weighted_recall_mean": weighted_recall_mean,
        "weighted_recall_std": weighted_recall_std,
        "labels": labels,
        "confusion_matrix": cm,
        "effective_splits": effective_splits,
        "final_class_dist": dict(class_counts),
    }


def run_supervised_kfold_eval(case_name_list, case_truth_label,
                              case_log_df, template_embedding,
                              n_splits=5, random_state=42,
                              min_samples_per_class=2,
                              rare_class_strategy="drop"):
    if len(case_name_list) != len(case_truth_label):
        raise ValueError("case_name_list and case_truth_label length mismatch")

    class_counts = Counter(case_truth_label.tolist())
    rare_labels = [label for label, count in class_counts.items() if count < min_samples_per_class]

    if rare_labels:
        if rare_class_strategy == "drop":
            keep_indices = [idx for idx, y in enumerate(case_truth_label.tolist()) if y not in rare_labels]
            dropped_count = len(case_name_list) - len(keep_indices)
            case_name_list = [case_name_list[idx] for idx in keep_indices]
            case_truth_label = case_truth_label[keep_indices]
            case_log_df = {name: case_log_df[name] for name in case_name_list}
            class_counts = Counter(case_truth_label.tolist())
            print(
                f"[Warn] Dropped {dropped_count} samples from rare classes "
                f"(labels={rare_labels}, min_samples_per_class={min_samples_per_class})."
            )
            print(f"[Info] Remaining class distribution: {class_counts}")
        else:
            raise ValueError(
                f"Not enough samples per class for StratifiedKFold: "
                f"rare_labels={rare_labels}, min_samples_per_class={min_samples_per_class}"
            )

    min_class_count = min(class_counts.values())
    effective_splits = min(n_splits, min_class_count)
    if effective_splits < 2:
        raise ValueError(
            f"Not enough samples per class for StratifiedKFold after filtering: "
            f"min_class_count={min_class_count}."
        )

    skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=random_state)

    acc_list, macro_f1_list = [], []
    weighted_f1_list, macro_recall_list, weighted_recall_list = [], [], []
    y_true_all, y_pred_all = [], []

    for fold_idx, (train_index, test_index) in enumerate(
        skf.split(np.arange(len(case_name_list)), case_truth_label),
        start=1,
    ):
        train_index = train_index.tolist()
        test_index = test_index.tolist()

        train_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in train_index}
        test_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in test_index}

        try:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding, embedding_size=EMBEDDING_SIZE)
        except TypeError:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding)

        model.get_train_embedding()
        model.get_test_embedding()

        train_set = np.array([model.train_embedding_dict[case_name_list[idx]] for idx in train_index])
        test_set = np.array([model.test_embedding_dict[case_name_list[idx]] for idx in test_index])

        y_train = case_truth_label[train_index].astype(int)
        y_true = case_truth_label[test_index].astype(int)

        classifier = clf(**clf_kwargs)
        classifier.fit(train_set, y_train)
        y_pred = classifier.predict(test_set).astype(int)

        m = _calc_eval_metrics(y_true, y_pred)
        acc_list.append(m["acc"])
        macro_f1_list.append(m["macro_f1"])
        weighted_f1_list.append(m["weighted_f1"])
        macro_recall_list.append(m["macro_recall"])
        weighted_recall_list.append(m["weighted_recall"])
        y_true_all.append(y_true)
        y_pred_all.append(y_pred)

        print(
            f"fold {fold_idx}/{effective_splits}: "
            f"acc={m['acc']:.4f}, macro_f1={m['macro_f1']:.4f}, weighted_f1={m['weighted_f1']:.4f}, "
            f"macro_recall={m['macro_recall']:.4f}, weighted_recall={m['weighted_recall']:.4f}, test_size={len(test_index)}"
        )

    y_true_all = np.concatenate(y_true_all).astype(int)
    y_pred_all = np.concatenate(y_pred_all).astype(int)

    acc_mean, acc_std = float(np.mean(acc_list)), float(np.std(acc_list))
    macro_f1_mean, macro_f1_std = float(np.mean(macro_f1_list)), float(np.std(macro_f1_list))
    weighted_f1_mean, weighted_f1_std = float(np.mean(weighted_f1_list)), float(np.std(weighted_f1_list))
    macro_recall_mean, macro_recall_std = float(np.mean(macro_recall_list)), float(np.std(macro_recall_list))
    weighted_recall_mean, weighted_recall_std = float(np.mean(weighted_recall_list)), float(np.std(weighted_recall_list))

    labels = np.unique(case_truth_label)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

    print("=" * 30, "Supervised StratifiedKFold Summary", "=" * 30)
    print(f"accuracy: mean={acc_mean:.4f}, std={acc_std:.4f}")
    print(f"macro_f1: mean={macro_f1_mean:.4f}, std={macro_f1_std:.4f}")
    print(f"weighted_f1: mean={weighted_f1_mean:.4f}, std={weighted_f1_std:.4f}")
    print(f"macro_recall: mean={macro_recall_mean:.4f}, std={macro_recall_std:.4f}")
    print(f"weighted_recall: mean={weighted_recall_mean:.4f}, std={weighted_recall_std:.4f}")
    print("confusion_matrix labels:", labels.tolist())
    print(cm)

    return {
        "acc_list": acc_list,
        "macro_f1_list": macro_f1_list,
        "f1_list": macro_f1_list,
        "weighted_f1_list": weighted_f1_list,
        "macro_recall_list": macro_recall_list,
        "weighted_recall_list": weighted_recall_list,
        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "macro_f1_mean": macro_f1_mean,
        "f1_mean": macro_f1_mean,
        "macro_f1_std": macro_f1_std,
        "f1_std": macro_f1_std,
        "weighted_f1_mean": weighted_f1_mean,
        "weighted_f1_std": weighted_f1_std,
        "macro_recall_mean": macro_recall_mean,
        "macro_recall_std": macro_recall_std,
        "weighted_recall_mean": weighted_recall_mean,
        "weighted_recall_std": weighted_recall_std,
        "labels": labels,
        "confusion_matrix": cm,
        "effective_splits": effective_splits,
        "final_class_dist": dict(class_counts),
    }





In [6]:
label_level_to_config = {
    "major": OS_CONFIG_MAJOR_PATH,
    "minor": OS_CONFIG_MINOR_PATH,
    "pair": OS_CONFIG_PAIR_PATH,
}

if LABEL_LEVEL not in label_level_to_config:
    raise ValueError(f"Unsupported LABEL_LEVEL: {LABEL_LEVEL}. Choose from {list(label_level_to_config.keys())}.")

config_path = label_level_to_config[LABEL_LEVEL]
case_dir = OS_CASE_DIR

if not os.path.exists(config_path):
    raise FileNotFoundError(f"config not found: {config_path}")
if not os.path.exists(case_dir):
    raise FileNotFoundError(f"case dir not found: {case_dir}")

case_name_list = sorted([os.path.splitext(name)[0] for name in os.listdir(case_dir) if name.endswith(".csv")])
config = json.load(open(config_path, "r", encoding="utf-8"))
case_truth_label = get_case_truth_label(case_name_list, config)

case_log_df = {}
for case_name in case_name_list:
    df_case = pd.read_csv(os.path.join(case_dir, case_name + ".csv"), dtype={"EventId": "string"})
    if "EventId" not in df_case.columns:
        raise ValueError(f"EventId column missing in case file: {case_name}.csv")
    df_case["EventId"] = df_case["EventId"].fillna("NO_LOG").astype(str)
    case_log_df[case_name] = df_case


def build_template_embedding(case_log_df, embedding_size=128, mode="hash_text", seed=42):
    if mode == "random":
        rng = np.random.default_rng(seed)
        all_templates = sorted(
            {
                str(eid)
                for df in case_log_df.values()
                for eid in df["EventId"].dropna().values
                if str(eid) != ""
            }
        )
        return {eid: rng.normal(0, 1, embedding_size) for eid in all_templates}

    eventid_to_template = {}
    for df in case_log_df.values():
        if "EventTemplate" in df.columns:
            for eid, tpl in zip(df["EventId"].astype(str).tolist(), df["EventTemplate"].astype(str).tolist()):
                if eid and eid not in eventid_to_template:
                    eventid_to_template[eid] = tpl
        else:
            for eid in df["EventId"].astype(str).tolist():
                if eid and eid not in eventid_to_template:
                    eventid_to_template[eid] = eid

    all_ids = sorted(eventid_to_template.keys())
    texts = [eventid_to_template[eid] for eid in all_ids]

    vectorizer = HashingVectorizer(
        n_features=embedding_size,
        alternate_sign=False,
        norm=None,
        token_pattern=r"(?u)\b\w+\b",
    )
    mat = vectorizer.transform(texts).toarray().astype(float)

    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    mat = mat / norms

    return {eid: mat[idx] for idx, eid in enumerate(all_ids)}


if template_embedding is None:
    template_embedding = build_template_embedding(
        case_log_df=case_log_df,
        embedding_size=EMBEDDING_SIZE,
        mode=TEMPLATE_EMBEDDING_MODE,
        seed=42,
    )
    print("template_embedding initialized:", len(template_embedding), "mode=", TEMPLATE_EMBEDDING_MODE)

print("DATASET:", "OS_preprocessed")
print("LABEL_LEVEL:", LABEL_LEVEL)
print("config_path:", config_path)
print("case_dir:", case_dir)
print("case_num:", len(case_name_list))
print("class_dist:", Counter(case_truth_label.tolist()))



template_embedding initialized: 16622 mode= random
DATASET: OS_preprocessed
LABEL_LEVEL: major
config_path: ..\data\OS_preprocessed\config_major.json
case_dir: ..\data\OS_preprocessed\cases
case_num: 500
class_dist: Counter({1: 272, 3: 117, 2: 55, 4: 43, 0: 11, 5: 2})


In [7]:
print('-' * 30, f"Eval Mode: {EVAL_MODE}", '-' * 30)
if EVAL_MODE == "supervised":
    eval_result = run_supervised_kfold_eval(
        case_name_list=case_name_list,
        case_truth_label=case_truth_label,
        case_log_df=case_log_df,
        template_embedding=template_embedding,
        n_splits=5,
        random_state=42,
        min_samples_per_class=MIN_SAMPLES_PER_CLASS,
        rare_class_strategy=RARE_CLASS_STRATEGY,
    )
elif EVAL_MODE == "logkg":
    eval_result = run_stratified_kfold_eval(
        case_name_list=case_name_list,
        case_truth_label=case_truth_label,
        case_log_df=case_log_df,
        template_embedding=template_embedding,
        n_splits=5,
        random_state=42,
        verbose_fold=False,
        min_samples_per_class=MIN_SAMPLES_PER_CLASS,
        rare_class_strategy=RARE_CLASS_STRATEGY,
    )
else:
    raise ValueError(f"Unsupported EVAL_MODE: {EVAL_MODE}")



------------------------------ Eval Mode: supervised ------------------------------
fold 1/2: acc=0.7200, macro_f1=0.4528, weighted_f1=0.6888, macro_recall=0.4073, weighted_recall=0.7200, test_size=250
fold 2/2: acc=0.6560, macro_f1=0.4760, weighted_f1=0.6462, macro_recall=0.4309, weighted_recall=0.6560, test_size=250
============================== Supervised StratifiedKFold Summary ==============================
accuracy: mean=0.6880, std=0.0320
macro_f1: mean=0.4644, std=0.0116
weighted_f1: mean=0.6675, std=0.0213
macro_recall: mean=0.4191, std=0.0118
weighted_recall: mean=0.6880, std=0.0320
confusion_matrix labels: [0, 1, 2, 3, 4, 5]
[[  4   5   0   2   0   0]
 [  0 225   4  42   1   0]
 [  0  23  20  11   1   0]
 [  0  28   2  85   2   0]
 [  0  27   1   5  10   0]
 [  0   1   0   0   1   0]]
